# 01 — OCR Structure & Quality

Objetivo deste notebook:

> Perceber, normalizar e caracterizar os ficheiros `*_ocr.pkl` dos telejornais.

Este notebook foca-se apenas em:

1. Estrutura dos pickles OCR.
2. Volume dos dados por canal/data.
3. Qualidade do OCR através das confidence scores.
4. Quantidade de texto ao longo do tempo.
5. Localização espacial do texto no ecrã.

Não inclui ainda análise política detalhada, comparação profunda RTP vs TVI, segmentação, speech ou polls.


## 0. Setup

Estrutura esperada:

```text
Big Data/
├── 01_ocr_structure_quality.ipynb
└── data/
    ├── features/
    │   ├── Telejornal_RTP_Dec_11_ocr.pkl
    │   ├── Telejornal_TVI_Dec_11_ocr.pkl
    │   └── ...
    └── frames/          # opcional, só necessário para visualizar frames
```


In [ ]:
from pathlib import Path
from collections import Counter
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

try:
    from PIL import Image, ImageDraw
    PIL_AVAILABLE = True
except Exception:
    PIL_AVAILABLE = False

BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
FEATURES_DIR = DATA_DIR / "features"
FRAMES_DIR = DATA_DIR / "frames"
OUTPUT_DIR = BASE_DIR / "outputs_ocr_01"

OUTPUT_DIR.mkdir(exist_ok=True)

print("BASE_DIR:", BASE_DIR.resolve())
print("FEATURES_DIR exists:", FEATURES_DIR.exists())
print("FRAMES_DIR exists:", FRAMES_DIR.exists())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())
print("PIL available:", PIL_AVAILABLE)


## 1. Descobrir ficheiros OCR

Primeiro, listamos todos os ficheiros `*_ocr.pkl` disponíveis.


In [ ]:
# 1. Obter e contar a lista de ficheiros OCR
ocr_files = sorted(FEATURES_DIR.glob("*_ocr.pkl"))
print("Número de ficheiros OCR:", len(ocr_files))

MONTH_ORDER = {
    "Nov": 11,
    "Dec": 12,
    "Jan": 13,   # Jan vem depois de Dec nesta timeline
}

def parse_ocr_filename(file_input):
    """
    Aceita tanto:
      - Path completo: data/features/Telejornal_RTP_Nov_10_ocr.pkl
      - apenas nome:  Telejornal_RTP_Nov_10_ocr.pkl

    Esperado:
      Telejornal_RTP_Nov_10_ocr.pkl
      Telejornal_TVI_Dec_11_ocr.pkl

    Devolve metadata sem obrigar o ficheiro a existir.
    """
    path = Path(file_input)
    filename = path.name
    stem = path.stem

    pattern = r"^Telejornal_(RTP|TVI)_([A-Za-z]+)_(\d+)_ocr$"
    match = re.match(pattern, stem)

    if match is None:
        channel = "unknown"
        month = "unknown"
        day = None
        date_label = "unknown"
        date_sort = 99999
    else:
        channel, month, day = match.groups()
        month = month[:3].title()
        day = int(day)
        date_label = f"{month}_{day:02d}"
        date_sort = MONTH_ORDER.get(month, 99) * 100 + day

    # Só calcula tamanho se recebeu um caminho que existe.
    # Isto evita FileNotFoundError quando passamos apenas f.name.
    size_kb = round(path.stat().st_size / 1024, 1) if path.exists() else np.nan

    return {
        "file": filename,
        "stem": stem,
        "channel": channel,
        "month": month,
        "day": day,
        "date_label": date_label,
        "date_sort": date_sort,
        "size_kb": size_kb,
    }

# 2. Criar o DataFrame com metadata dos ficheiros
file_meta = pd.DataFrame([parse_ocr_filename(f) for f in ocr_files])

if not file_meta.empty:
    file_meta = file_meta.sort_values(["date_sort", "channel"]).reset_index(drop=True)

display(file_meta)

file_meta.to_csv(OUTPUT_DIR / "ocr_file_metadata.csv", index=False)
print("Saved to:", OUTPUT_DIR / "ocr_file_metadata.csv")


## 2. Carregar pickles e inspecionar estrutura

Aqui verificamos quantas linhas/colunas tem cada pickle e quais são as colunas disponíveis.


In [ ]:
ocr_raw = {}

for f in ocr_files:
    try:
        df = pd.read_pickle(f)
        ocr_raw[f.name] = df
        print(f.name, "->", df.shape, "| columns:", list(df.columns))
    except Exception as e:
        print("Erro ao ler", f.name, ":", repr(e))

print("\nTotal carregado:", len(ocr_raw))


In [ ]:
raw_summary = []

for file_name, df in ocr_raw.items():
    meta = parse_ocr_filename(file_name)

    raw_summary.append({
        "file": file_name,
        "channel": meta["channel"],
        "month": meta["month"],
        "day": meta["day"],
        "date_label": meta["date_label"],
        "date_sort": meta["date_sort"],
        "raw_rows": df.shape[0],
        "raw_columns": df.shape[1],
        "columns": ", ".join(map(str, df.columns)),
        "memory_MB": round(df.memory_usage(deep=True).sum() / 1024**2, 3),
    })

raw_summary_df = pd.DataFrame(raw_summary)

if not raw_summary_df.empty:
    raw_summary_df = (
        raw_summary_df
        .sort_values(["date_sort", "channel"])
        .reset_index(drop=True)
    )

display(raw_summary_df)

raw_summary_df.to_csv(OUTPUT_DIR / "raw_pickle_summary.csv", index=False)

print("Saved to:", OUTPUT_DIR / "raw_pickle_summary.csv")


### 2.1 Exemplo de um pickle

Esta célula ajuda a perceber o formato interno. Normalmente esperamos:

- `Frame`: caminho ou identificador do frame;
- `OCR`: lista de deteções OCR nesse frame;
- cada deteção OCR contém `text`, `bbox` e `conf`.


In [ ]:
example_file = sorted(ocr_raw.keys())[0]
df_example = ocr_raw[example_file]

print("Exemplo:", example_file)
print("Shape:", df_example.shape)
print("Columns:", list(df_example.columns))

display(df_example.head(5))

print("\nPrimeira linha, com tipos:")
row0 = df_example.iloc[0]
for col in df_example.columns:
    value = row0[col]
    print("\nCOLUNA:", col)
    print("TIPO:", type(value))
    value_str = str(value)
    print(value_str[:1200] + (" ..." if len(value_str) > 1200 else ""))


## 3. Normalizar OCR

Vamos transformar todos os pickles numa única tabela:

```text
uma linha = uma deteção textual num frame
```

Campos principais:

- `file`
- `channel`
- `date_label`
- `frame`
- `minute`
- `text`
- `confidence`
- `bbox`
- `x1`, `y1`, `x2`, `y2`


In [ ]:
def safe_float(x):
    try:
        if x is None:
            return np.nan
        return float(x)
    except Exception:
        return np.nan

def is_number_like(x):
    try:
        float(x)
        return True
    except Exception:
        return False

def extract_frame_number(frame_value, fallback_index):
    """
    Se Frame for caminho tipo frame_1956.jpg, extrai 1956.
    Se não der, usa o índice da linha.
    """
    if isinstance(frame_value, (str, Path)):
        nums = re.findall(r"\d+", Path(str(frame_value)).stem)
        if nums:
            return int(nums[-1])
        return int(fallback_index)

    if is_number_like(frame_value):
        return int(float(frame_value))

    return int(fallback_index)

def extract_text(det):
    if isinstance(det, dict):
        for key in ["text", "Text", "word", "value", "label"]:
            if key in det:
                return str(det[key])
    if isinstance(det, str):
        return det
    return None

def extract_conf(det):
    if isinstance(det, dict):
        for key in ["conf", "confidence", "score", "prob", "probability"]:
            if key in det:
                return safe_float(det[key])
    return np.nan

def extract_bbox(det):
    if not isinstance(det, dict):
        return None

    for key in ["bbox", "box", "location", "locations", "bounding_box"]:
        if key in det:
            b = det[key]
            if isinstance(b, (list, tuple, np.ndarray)) and len(b) == 4:
                return [safe_float(v) for v in b]
    return None

def normalize_ocr_df(df, file_name):
    meta = parse_ocr_filename(file_name)
    rows = []

    frame_col = "Frame" if "Frame" in df.columns else df.columns[0]
    ocr_col = "OCR" if "OCR" in df.columns else None

    if ocr_col is None:
        raise ValueError(f"Não encontrei coluna OCR em {file_name}. Colunas: {list(df.columns)}")

    for idx, row in df.iterrows():
        frame_original = row[frame_col]
        frame_number = extract_frame_number(frame_original, idx)
        detections = row[ocr_col]

        if not isinstance(detections, (list, tuple, np.ndarray)):
            continue

        for det in detections:
            text = extract_text(det)
            if text is None or len(str(text).strip()) == 0:
                continue

            bbox = extract_bbox(det)
            conf = extract_conf(det)

            rows.append({
                "file": file_name,
                "channel": meta["channel"],
                "month": meta["month"],
                "day": meta["day"],
                "date_label": meta["date_label"],
                "date_sort": meta["date_sort"],
                "frame": frame_number,
                "frame_original": str(frame_original),
                "text": str(text),
                "confidence": conf,
                "bbox": bbox,
            })

    out = pd.DataFrame(rows)

    if out.empty:
        return out

    out["second"] = out["frame"]
    out["minute"] = (out["second"] // 60).astype(int)

    def bbox_value(b, i):
        if isinstance(b, (list, tuple, np.ndarray)) and len(b) == 4:
            return safe_float(b[i])
        return np.nan

    out["x1"] = out["bbox"].apply(lambda b: bbox_value(b, 0))
    out["y1"] = out["bbox"].apply(lambda b: bbox_value(b, 1))
    out["x2"] = out["bbox"].apply(lambda b: bbox_value(b, 2))
    out["y2"] = out["bbox"].apply(lambda b: bbox_value(b, 3))
    out["bbox_width"] = (out["x2"] - out["x1"]).clip(lower=0)
    out["bbox_height"] = (out["y2"] - out["y1"]).clip(lower=0)
    out["bbox_area"] = out["bbox_width"] * out["bbox_height"]

    return out


In [ ]:
ocr_long_parts = []

for file_name, df in ocr_raw.items():
    norm = normalize_ocr_df(df, file_name)
    print(file_name, "-> deteções normalizadas:", len(norm))
    ocr_long_parts.append(norm)

ocr_long = pd.concat(ocr_long_parts, ignore_index=True) if ocr_long_parts else pd.DataFrame()

print("\nTabela normalizada:", ocr_long.shape)
display(ocr_long.head(10))

ocr_long.to_pickle(OUTPUT_DIR / "ocr_long_normalized.pkl")
ocr_long.to_csv(OUTPUT_DIR / "ocr_long_normalized.csv", index=False)


## 4. Limpeza mínima do texto

Aqui criamos colunas úteis:

- `clean_text`
- `n_chars`
- `n_words`

Removemos tags HTML simples, como `<b>` e `<br>`, que aparecem em alguns OCRs.


In [ ]:
STOPWORDS_PT = {
    "de","a","o","e","que","do","da","em","um","uma","para","com","não","os","as","no","na","por",
    "se","ao","dos","das","mais","como","é","foi","são","ser","tem","também","ou","à","às","nos",
    "nas","sobre","entre","até","sem","já","lhe","ele","ela","eles","elas","sua","seu","suas","seus",
    "este","esta","estes","estas","isso","isto","há","vai","ter","mas","muito","muita","muitos","muitas",
    "porque","quando","onde","quem","qual","quais","todo","toda","todos","todas","num","numa","pelo","pela",
    "pelos","pelas","aos","ainda","só","era","foram","será","serem"
}

def clean_text_pt(text):
    text = str(text).lower()
    text = re.sub(r"<[^>]+>", " ", text)       # remove tags HTML
    text = text.replace("\\n", " ")
    text = re.sub(r"[^a-záàâãéèêíóôõúç0-9\s]", " ", text)
    text = re.sub(r"\b(b|br)\b", " ", text)    # restos de tags
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_pt(text, remove_stopwords=False, min_len=2):
    tokens = clean_text_pt(text).split()
    tokens = [t for t in tokens if len(t) >= min_len]
    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOPWORDS_PT]
    return tokens

ocr_long["clean_text"] = ocr_long["text"].apply(clean_text_pt)
ocr_long["n_chars"] = ocr_long["clean_text"].str.len()
ocr_long["n_words"] = ocr_long["clean_text"].apply(lambda x: len(tokenize_pt(x, remove_stopwords=False)))

display(ocr_long[["text", "clean_text", "confidence", "frame", "minute"]].head(20))


### 4.1 Aplicar limpeza ao OCR normalizado

A célula anterior só definia funções. Esta célula cria efetivamente as colunas usadas nas análises seguintes:

- `clean_text`
- `tokens`
- `tokens_no_stopwords`
- `n_chars`
- `n_words`


In [ ]:
# Aplicar limpeza ao texto OCR normalizado
if ocr_long.empty:
    print("ocr_long está vazio. Verifica se os pickles foram carregados e normalizados corretamente.")
else:
    ocr_long["clean_text"] = ocr_long["text"].apply(clean_text_pt)
    ocr_long["tokens"] = ocr_long["clean_text"].apply(lambda x: tokenize_pt(x, remove_stopwords=False))
    ocr_long["tokens_no_stopwords"] = ocr_long["clean_text"].apply(lambda x: tokenize_pt(x, remove_stopwords=True))
    ocr_long["n_chars"] = ocr_long["clean_text"].str.len()
    ocr_long["n_words"] = ocr_long["tokens"].apply(len)

    display(ocr_long[["file", "frame", "minute", "text", "clean_text", "confidence", "n_words"]].head(10))

    ocr_long.to_pickle(OUTPUT_DIR / "ocr_long_normalized_with_text_features.pkl")
    ocr_long.to_csv(OUTPUT_DIR / "ocr_long_normalized_with_text_features.csv", index=False)

    print("Saved cleaned OCR table to:")
    print("-", OUTPUT_DIR / "ocr_long_normalized_with_text_features.pkl")
    print("-", OUTPUT_DIR / "ocr_long_normalized_with_text_features.csv")


## 5. Estrutura e volume dos dados OCR

Aqui resumimos volume por telejornal:

- frames com OCR;
- deteções OCR;
- duração aproximada;
- deteções por frame;
- palavras OCR;
- confidence média/mediana.


In [ ]:
volume_df = ocr_long.groupby(["file", "channel", "date_label", "date_sort"]).agg(
    detections=("text", "size"),
    frames_with_ocr=("frame", "nunique"),
    first_frame=("frame", "min"),
    last_frame=("frame", "max"),
    approx_duration_min=("minute", "max"),
    total_words=("n_words", "sum"),
    avg_words_per_detection=("n_words", "mean"),
    avg_confidence=("confidence", "mean"),
    median_confidence=("confidence", "median"),
    low_conf_ratio=("confidence", lambda s: (s < 0.5).mean()),
    bbox_available_ratio=("bbox", lambda s: s.notna().mean())
).reset_index()

volume_df["detections_per_frame"] = volume_df["detections"] / volume_df["frames_with_ocr"]

volume_df = volume_df.sort_values(["date_sort", "channel"]).reset_index(drop=True)

display(volume_df)

volume_df.to_csv(OUTPUT_DIR / "ocr_volume_quality_by_file.csv", index=False)
print("Saved to:", OUTPUT_DIR / "ocr_volume_quality_by_file.csv")


In [ ]:
display(Markdown("### Resumo global"))

if ocr_long.empty:
    print("ocr_long está vazio.")
else:
    global_summary = pd.Series({
        "n_ocr_files": ocr_long["file"].nunique(),
        "channels": ", ".join(sorted(ocr_long["channel"].dropna().unique())),
        "n_dates": ocr_long["date_label"].nunique(),
        "total_ocr_detections": len(ocr_long),
        "total_frames_with_ocr": ocr_long[["file", "frame"]].drop_duplicates().shape[0],
        "avg_confidence": ocr_long["confidence"].mean(),
        "median_confidence": ocr_long["confidence"].median(),
        "low_conf_ratio_<0.5": (ocr_long["confidence"] < 0.5).mean(),
        "bbox_available_ratio": ocr_long["bbox"].notna().mean(),
    })

    display(global_summary)


In [ ]:
# Plot: deteções por telejornal
plot_df = volume_df.sort_values(["date_sort", "channel"]).copy()
plot_df["label"] = plot_df["channel"] + " " + plot_df["date_label"]

plt.figure(figsize=(12, 4))
plt.bar(plot_df["label"], plot_df["detections"])
plt.title("Total de deteções OCR por telejornal")
plt.ylabel("Nº deteções OCR")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Plot: frames com OCR
plt.figure(figsize=(12, 4))
plt.bar(plot_df["label"], plot_df["frames_with_ocr"])
plt.title("Frames com OCR por telejornal")
plt.ylabel("Nº frames com OCR")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Plot: duração aproximada
plt.figure(figsize=(12, 4))
plt.bar(plot_df["label"], plot_df["approx_duration_min"])
plt.title("Duração aproximada por telejornal")
plt.ylabel("Minuto final aproximado")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 6. Qualidade do OCR

A confidence score permite avaliar a qualidade das deteções.

Vamos observar:

- distribuição global;
- confidence por canal;
- confidence média por telejornal;
- exemplos de baixa confidence;
- exemplos de alta confidence.


In [ ]:
display(ocr_long["confidence"].describe())

plt.figure(figsize=(8, 4))
ocr_long["confidence"].dropna().hist(bins=50)
plt.title("Distribuição global da confidence do OCR")
plt.xlabel("Confidence")
plt.ylabel("Frequência")
plt.tight_layout()
plt.show()


In [ ]:
# Boxplot por canal
ocr_long.boxplot(column="confidence", by="channel", figsize=(7, 4))
plt.title("Confidence OCR por canal")
plt.suptitle("")
plt.xlabel("Canal")
plt.ylabel("Confidence")
plt.tight_layout()
plt.show()

# Confidence média por telejornal
plt.figure(figsize=(12, 4))
plt.bar(plot_df["label"], plot_df["avg_confidence"])
plt.title("Confidence média por telejornal")
plt.ylabel("Confidence média")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
display(Markdown("### Exemplos com baixa confidence"))
low_conf_examples = ocr_long.sort_values("confidence", ascending=True)[
    ["file", "frame", "minute", "text", "clean_text", "confidence"]
].head(20)
display(low_conf_examples)

display(Markdown("### Exemplos com alta confidence"))
high_conf_examples = ocr_long.sort_values("confidence", ascending=False)[
    ["file", "frame", "minute", "text", "clean_text", "confidence"]
].head(20)
display(high_conf_examples)

low_conf_examples.to_csv(OUTPUT_DIR / "low_confidence_examples.csv", index=False)
high_conf_examples.to_csv(OUTPUT_DIR / "high_confidence_examples.csv", index=False)


## 7. Quantidade de texto ao longo do tempo

Como os frames foram amostrados a 1 frame por segundo, podemos agregar por minuto.

Esta análise ajuda a detetar momentos com muito texto no ecrã, por exemplo:
- títulos;
- rodapés;
- gráficos;
- blocos com mais informação visual.


In [ ]:
per_minute = ocr_long.groupby(["file", "channel", "date_label", "date_sort", "minute"]).agg(
    n_detections=("text", "size"),
    n_words=("n_words", "sum"),
    avg_confidence=("confidence", "mean"),
    frames_with_ocr=("frame", "nunique"),
    text_concat=("clean_text", lambda s: " ".join(s))
).reset_index()

per_minute = per_minute.sort_values(["date_sort", "channel", "minute"]).reset_index(drop=True)

display(per_minute.head(20))

per_minute.to_csv(OUTPUT_DIR / "ocr_per_minute.csv", index=False)
print("Saved to:", OUTPUT_DIR / "ocr_per_minute.csv")


In [ ]:
# Escolhe aqui o telejornal para inspeção temporal
chosen_file = sorted(ocr_long["file"].unique())[0]
print("chosen_file =", chosen_file)

timeline = per_minute[per_minute["file"] == chosen_file].sort_values("minute")

plt.figure(figsize=(12, 4))
plt.plot(timeline["minute"], timeline["n_detections"], marker="o")
plt.title(f"Deteções OCR por minuto — {chosen_file}")
plt.xlabel("Minuto")
plt.ylabel("Nº deteções OCR")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(timeline["minute"], timeline["n_words"], marker="o")
plt.title(f"Palavras OCR por minuto — {chosen_file}")
plt.xlabel("Minuto")
plt.ylabel("Nº palavras OCR")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(timeline["minute"], timeline["avg_confidence"], marker="o")
plt.title(f"Confidence média por minuto — {chosen_file}")
plt.xlabel("Minuto")
plt.ylabel("Confidence média")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Minutos com mais texto no telejornal escolhido
top_minutes = timeline.sort_values("n_detections", ascending=False).head(10)
display(top_minutes[["file", "minute", "n_detections", "n_words", "avg_confidence", "text_concat"]])


## 8. Localização espacial do texto no ecrã

Cada deteção OCR tem uma bounding box.  
Isto permite estudar onde aparece mais texto:

- `bottom_*`: rodapés, nomes, legendas;
- `top_*`: logos, direto, hora;
- `center`: gráficos, títulos grandes, imagens com texto.


In [ ]:
bbox_df = ocr_long.dropna(subset=["x1", "y1", "x2", "y2"]).copy()

print("Deteções com bounding box:", len(bbox_df), "de", len(ocr_long))
print("Percentagem:", round(len(bbox_df) / max(len(ocr_long), 1), 3))

bbox_df["cx"] = (bbox_df["x1"] + bbox_df["x2"]) / 2
bbox_df["cy"] = (bbox_df["y1"] + bbox_df["y2"]) / 2

display(bbox_df[["file", "frame", "text", "confidence", "x1", "y1", "x2", "y2", "cx", "cy", "bbox_area"]].head())


In [ ]:
if len(bbox_df) > 0:
    plt.figure(figsize=(6, 5))
    plt.scatter(bbox_df["cx"], bbox_df["cy"], s=4, alpha=0.25)
    plt.gca().invert_yaxis()
    plt.title("Centros das bounding boxes OCR")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(6, 5))
    plt.hist2d(bbox_df["cx"], bbox_df["cy"], bins=40)
    plt.gca().invert_yaxis()
    plt.title("Heatmap espacial das deteções OCR")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.colorbar(label="Nº deteções")
    plt.tight_layout()
    plt.show()


In [ ]:
# Definir zonas do ecrã com base nos percentis altos das coordenadas
if len(bbox_df) > 0:
    max_x = bbox_df["x2"].quantile(0.99)
    max_y = bbox_df["y2"].quantile(0.99)

    def zone_x(cx):
        if cx < max_x / 3:
            return "left"
        elif cx < 2 * max_x / 3:
            return "center"
        return "right"

    def zone_y(cy):
        if cy < max_y / 3:
            return "top"
        elif cy < 2 * max_y / 3:
            return "middle"
        return "bottom"

    bbox_df["zone_x"] = bbox_df["cx"].apply(zone_x)
    bbox_df["zone_y"] = bbox_df["cy"].apply(zone_y)
    bbox_df["screen_zone"] = bbox_df["zone_y"] + "_" + bbox_df["zone_x"]

    zone_summary = bbox_df.groupby("screen_zone").agg(
        detections=("text", "size"),
        avg_confidence=("confidence", "mean"),
        median_confidence=("confidence", "median"),
        avg_area=("bbox_area", "mean")
    ).reset_index().sort_values("detections", ascending=False)

    display(zone_summary)

    zone_summary.to_csv(OUTPUT_DIR / "ocr_screen_zone_summary.csv", index=False)

    plt.figure(figsize=(10, 4))
    plt.bar(zone_summary["screen_zone"], zone_summary["detections"])
    plt.title("Distribuição das deteções OCR por zona do ecrã")
    plt.ylabel("Nº deteções")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    bbox_df.boxplot(column="confidence", by="screen_zone", figsize=(10, 4))
    plt.title("Confidence por zona do ecrã")
    plt.suptitle("")
    plt.xlabel("Zona do ecrã")
    plt.ylabel("Confidence")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


## 9. Visualização opcional de uma frame com OCR

Esta secção só funciona se tiveres os frames dos telejornais em `data/frames`.

Se não tiveres os frames no portátil, podes ignorar esta parte.  
A análise dos pickles continua válida sem os frames.


In [ ]:
def find_frame_image(file_name, frame_number):
    """
    Tenta encontrar a imagem correspondente ao frame.
    Usa primeiro o caminho original guardado no pickle.
    Depois procura dentro de data/frames pelo mesmo nome de ficheiro.
    """
    rows = ocr_long[(ocr_long["file"] == file_name) & (ocr_long["frame"] == frame_number)]
    if not rows.empty:
        original = rows.iloc[0]["frame_original"]
        p = Path(original)

        if p.exists():
            return p

        matches = list(FRAMES_DIR.rglob(p.name))
        if matches:
            return matches[0]

    stem = file_name.replace("_ocr.pkl", "")
    possible_dir = FRAMES_DIR / stem
    if possible_dir.exists():
        patterns = [
            f"*{int(frame_number)}*.jpg",
            f"*{int(frame_number):04d}*.jpg",
            f"*{int(frame_number):05d}*.jpg",
            f"*{int(frame_number):06d}*.jpg",
            f"*{int(frame_number)}*.png",
            f"*{int(frame_number):04d}*.png",
            f"*{int(frame_number):05d}*.png",
            f"*{int(frame_number):06d}*.png",
        ]
        for pattern in patterns:
            matches = list(possible_dir.rglob(pattern))
            if matches:
                return matches[0]

    return None

def draw_ocr_on_frame(file_name, frame_number, max_boxes=40):
    if not PIL_AVAILABLE:
        print("PIL não está disponível.")
        return

    img_path = find_frame_image(file_name, frame_number)

    if img_path is None:
        print("Não encontrei a imagem local para:")
        print("file:", file_name)
        print("frame:", frame_number)
        print("\nIsto é normal se não tiveres os frames dos telejornais neste computador.")
        return

    img = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(img)

    dets = ocr_long[(ocr_long["file"] == file_name) & (ocr_long["frame"] == frame_number)]
    dets = dets.dropna(subset=["x1", "y1", "x2", "y2"]).head(max_boxes)

    for _, r in dets.iterrows():
        box = [r["x1"], r["y1"], r["x2"], r["y2"]]
        draw.rectangle(box, outline="red", width=2)
        draw.text((r["x1"], max(r["y1"] - 12, 0)), str(r["clean_text"])[:30], fill="red")

    plt.figure(figsize=(14, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"{file_name} — frame {frame_number}")
    plt.show()

# Escolher uma frame com muitas deteções OCR
frame_counts = ocr_long.groupby(["file", "frame"]).size().reset_index(name="n_detections")
frame_counts = frame_counts.sort_values("n_detections", ascending=False)

display(frame_counts.head(10))

test_file = frame_counts.iloc[0]["file"]
test_frame = frame_counts.iloc[0]["frame"]

print("Tentativa de visualização:")
print(test_file, test_frame)

draw_ocr_on_frame(test_file, test_frame)


## 10. Notas preliminares

Depois de correres o notebook, preenche esta secção com as tuas conclusões.

Pontos a observar:

- Quantos OCR pickles existem?
- Há diferenças claras entre RTP e TVI em volume de texto?
- A confidence média é alta ou baixa?
- Há muitos exemplos de OCR com baixa confidence?
- Em que zonas do ecrã aparece mais texto?
- Os picos de texto por minuto parecem indicar blocos com mais informação visual?


In [ ]:
observations = [
    "TODO: preencher após observar os outputs.",
]

for obs in observations:
    print("-", obs)


## 11. Outputs gerados

Este notebook guarda ficheiros úteis em:

```text
outputs_ocr_01/
```

Principais outputs:

- `ocr_file_metadata.csv`
- `raw_pickle_summary.csv`
- `ocr_long_normalized.csv`
- `ocr_volume_quality_by_file.csv`
- `ocr_per_minute.csv`
- `ocr_screen_zone_summary.csv`
- `low_confidence_examples.csv`
- `high_confidence_examples.csv`
